# Build multimodal MuData

Load the total RNA h5ad (from NB1) and per-embryo exon/intron RDS files,
apply gene filtering per modality, and build a 3-modality MuData (rna, spliced, unspliced).

**Memory-optimized approach**: Filter RNA first to free ~240GB before loading RDS files,
then load exon and intron sequentially.

**Inputs**:
- `gastrula_to_pup_rna_qc.h5ad` (from NB1)
- `scrublet_scores.csv` (from NB1a, optional)
- `embryo_exon_intron/*.rds` (148 files)

**Output**: `gastrula_to_pup_filtered.h5mu`

In [ ]:
import gc
import os
import time

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig = plt.figure()
plt.close(fig)  # force backend + C extensions before rds2py hooks imports

import anndata as ad  # noqa: E402
import mudata as mu  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import psutil  # noqa: E402
import scanpy as sc  # noqa: E402
from rds2py import read_rds  # noqa: E402

from regularizedvi.utils import filter_genes  # noqa: E402

DATA_DIR = "/nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup"
rds_dir = os.path.join(DATA_DIR, "embryo_exon_intron")

## 1. Load RNA data and Scrublet scores

In [2]:
h5ad_path = os.path.join(DATA_DIR, "gastrula_to_pup_rna_qc.h5ad")
print(f"Loading {h5ad_path}...")
adata = sc.read_h5ad(h5ad_path)
print(f"Shape: {adata.shape}, dtype: {adata.X.dtype}")
print(f"RSS after load: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

# Load Scrublet scores (optional — NB1a may still be running)
scrublet_csv = os.path.join(DATA_DIR, "scrublet_scores.csv")
if os.path.exists(scrublet_csv):
    scores = pd.read_csv(scrublet_csv, index_col="cell_id")
    adata.obs["doublet_score"] = scores["doublet_score"].reindex(adata.obs_names).values
    adata.obs["predicted_doublet"] = scores["predicted_doublet"].reindex(adata.obs_names).fillna(False).values
    print(f"Scrublet scores loaded: {scores.index.isin(adata.obs_names).sum():,} / {adata.n_obs:,} cells")
    del scores
else:
    print(f"Scrublet scores not found at {scrublet_csv} — skipping (NB1a not yet complete)")

Loading /nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup/gastrula_to_pup_rna_qc.h5ad...


Shape: (11441407, 24552), dtype: uint16
RSS after load: 188.2 GB


/tmp/ipykernel_2165669/1261676668.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  scores["predicted_doublet"].reindex(adata.obs_names).fillna(False).values


Scrublet scores loaded: 7,902,708 / 11,441,407 cells


## 2. Filter RNA genes

Filter RNA to shared genes (intersection with exon/intron), apply gene selection,
then free the full adata to reclaim ~240GB before loading RDS files.

In [3]:
# Get shared genes between RNA and exon/intron (one RDS file suffices — all use same gene ref)
_exon = read_rds(os.path.join(rds_dir, "embryo_1_exp_exon.rds"))
exon_var_names = pd.Index(list(_exon.dimnames[0]))
del _exon
gc.collect()

shared_genes = adata.var_names.intersection(exon_var_names)
print(f"Shared genes: {len(shared_genes)} / {adata.n_vars} (RNA) / {len(exon_var_names)} (exon)")

# Unified gene filtering cutoffs for all modalities
FILTER_KWARGS = {"cell_count_cutoff": 30, "cell_percentage_cutoff2": 0.005, "nonz_mean_cutoff": 1.05}

# Filter RNA to shared genes, then gene selection
adata_shared = adata[:, shared_genes]  # view, no copy
selected_rna = filter_genes(adata_shared, **FILTER_KWARGS)
adata_rna = adata_shared[:, selected_rna].copy()
print(f"RNA: {adata.n_vars} -> {len(shared_genes)} (shared) -> {adata_rna.n_vars} (filtered)")

# Save obs + cell names before freeing full adata
shared_obs = adata.obs.copy()
rna_cells = adata.obs_names.copy()
del adata, adata_shared
gc.collect()
print(f"RSS after RNA filter+free: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

Shared genes: 24552 / 24552 (RNA) / 49585 (exon)


/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/src/regularizedvi/utils/_filtering.py:76: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = n_cells


/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


RNA: 24552 -> 24552 (shared) -> 21619 (filtered)


RSS after RNA filter+free: 189.7 GB


## 3. Load exon (spliced)

In [4]:
spliced_adatas = []
t0 = time.time()
for i in range(1, 75):
    if i % 10 == 0:
        print(f"  Exon embryo {i}/74, RSS={psutil.Process().memory_info().rss / 1e9:.1f}GB")
    obj = read_rds(os.path.join(rds_dir, f"embryo_{i}_exp_exon.rds"))
    spliced_adatas.append(
        ad.AnnData(
            X=obj.matrix.T.tocsr().astype(np.uint16),
            obs=pd.DataFrame(index=list(obj.dimnames[1])),
            var=pd.DataFrame(index=list(obj.dimnames[0])),
        )
    )
    del obj
    gc.collect()

print(f"Loaded 74 exon files in {time.time() - t0:.0f}s")
adata_spliced = ad.concat(spliced_adatas, join="outer")
del spliced_adatas
gc.collect()
print(f"Spliced concat: {adata_spliced.shape}, dtype={adata_spliced.X.dtype}")
print(f"RSS after concat: {psutil.Process().memory_info().rss / 1e9:.1f}GB")

# Align to shared cells and genes
shared_cells = rna_cells.intersection(adata_spliced.obs_names)
print(f"Shared cells: {len(shared_cells)} / {len(rna_cells)} (RNA) / {adata_spliced.n_obs} (spliced)")
adata_spliced = adata_spliced[shared_cells, shared_genes].copy()
gc.collect()

# Gene filtering (same cutoffs as RNA)
selected_spliced = filter_genes(adata_spliced, **FILTER_KWARGS)
adata_spliced_filt = adata_spliced[:, selected_spliced].copy()
print(f"Spliced: {adata_spliced.n_vars} -> {adata_spliced_filt.n_vars} genes")
del adata_spliced
gc.collect()
print(f"RSS after exon filter: {psutil.Process().memory_info().rss / 1e9:.1f}GB")

  Exon embryo 10/74, RSS=190.2GB


  Exon embryo 20/74, RSS=192.8GB


  Exon embryo 30/74, RSS=199.0GB


  Exon embryo 40/74, RSS=209.7GB


  Exon embryo 50/74, RSS=217.4GB


...

Loaded 74 exon files in 796s


Spliced concat: (11441407, 49585), dtype=uint16
RSS after concat: 285.5GB


Shared cells: 9348879 / 11441407 (RNA) / 11441407 (spliced)


/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


Spliced: 24552 -> 20123 genes
RSS after exon filter: 266.4GB


## 4. Load intron (unspliced)

In [5]:
unspliced_adatas = []
t0 = time.time()
for i in range(1, 75):
    if i % 10 == 0:
        print(f"  Intron embryo {i}/74, RSS={psutil.Process().memory_info().rss / 1e9:.1f}GB")
    obj = read_rds(os.path.join(rds_dir, f"embryo_{i}_exp_intron.rds"))
    unspliced_adatas.append(
        ad.AnnData(
            X=obj.matrix.T.tocsr().astype(np.uint16),
            obs=pd.DataFrame(index=list(obj.dimnames[1])),
            var=pd.DataFrame(index=list(obj.dimnames[0])),
        )
    )
    del obj
    gc.collect()

print(f"Loaded 74 intron files in {time.time() - t0:.0f}s")
adata_unspliced = ad.concat(unspliced_adatas, join="outer")
del unspliced_adatas
gc.collect()
print(f"Unspliced concat: {adata_unspliced.shape}, dtype={adata_unspliced.X.dtype}")
print(f"RSS after concat: {psutil.Process().memory_info().rss / 1e9:.1f}GB")

# Align to shared cells and genes
adata_unspliced = adata_unspliced[shared_cells, shared_genes].copy()
gc.collect()

# Gene filtering (same cutoffs as RNA)
selected_unspliced = filter_genes(adata_unspliced, **FILTER_KWARGS)
adata_unspliced_filt = adata_unspliced[:, selected_unspliced].copy()
print(f"Unspliced: {adata_unspliced.n_vars} -> {adata_unspliced_filt.n_vars} genes")
del adata_unspliced
gc.collect()
print(f"RSS after intron filter: {psutil.Process().memory_info().rss / 1e9:.1f}GB")

  Intron embryo 10/74, RSS=267.2GB


  Intron embryo 20/74, RSS=270.1GB


  Intron embryo 30/74, RSS=277.3GB


  Intron embryo 40/74, RSS=292.3GB


  Intron embryo 50/74, RSS=303.0GB


...

  Intron embryo 70/74, RSS=327.3GB


Loaded 74 intron files in 1036s


Unspliced concat: (11441407, 49585), dtype=uint16
RSS after concat: 392.3GB


/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


Unspliced: 24552 -> 18437 genes
RSS after intron filter: 367.4GB


## 5. Build and save MuData

In [6]:
# Also subset RNA to shared_cells (in case exon/intron had fewer cells)
adata_rna = adata_rna[shared_cells].copy()

# Assign shared obs (QC, scrublet, pcr_well, embryo_id, etc.) to all modalities
obs_shared = shared_obs.reindex(shared_cells).copy()
adata_rna.obs = obs_shared.copy()
adata_spliced_filt.obs = obs_shared.copy()
adata_unspliced_filt.obs = obs_shared.copy()
del shared_obs, obs_shared

mdata = mu.MuData(
    {
        "rna": adata_rna,
        "spliced": adata_spliced_filt,
        "unspliced": adata_unspliced_filt,
    }
)
print(f"MuData: {mdata}")
for mod in mdata.mod:
    print(f"  {mod}: {mdata[mod].shape}, dtype={mdata[mod].X.dtype}")

h5mu_path = os.path.join(DATA_DIR, "gastrula_to_pup_filtered.h5mu")
mdata.write_h5mu(h5mu_path)
print(f"\nSaved: {h5mu_path} ({os.path.getsize(h5mu_path) / 1e9:.1f} GB)")
print(f"RSS final: {psutil.Process().memory_info().rss / 1e9:.1f} GB")

del mdata, adata_rna, adata_spliced_filt, adata_unspliced_filt
gc.collect()

/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/mudata/_core/mudata.py:571: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  self._update_attr_legacy(attr, axis, join_common, **kwargs)
/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become th

MuData: MuData object with n_obs × n_vars = 9348879 × 60179
  3 modalities
    rna:	9348879 x 21619
      obs:	'cell_id', 'keep', 'day', 'embryo_id', 'experimental_batch', 'major_trajectory', 'celltype_update', 'pcr_well', 'run_id', 'embryo_experiment', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'doublet_score', 'predicted_doublet'
      var:	'gene_id', 'gene_type', 'gene_short_name', 'chr', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'nonz_mean'
    spliced:	9348879 x 20123
      obs:	'cell_id', 'keep', 'day', 'embryo_id', 'experimental_batch', 'major_trajectory', 'celltype_update', 'pcr_well', 'run_id', 'embryo_experiment', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_t

/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/software/conda/users/vk7/regularizedvi/lib/python3.11/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)



Saved: /nfs/team205/vk7/sanger_projects/large_data/gastrula_to_pup/gastrula_to_pup_filtered.h5mu (340.3 GB)
RSS final: 528.7 GB


2923